# ModernBERT Routed 46-Head Training (CoNLL Native)

This notebook trains from canonical CoNLL under `cdatasets/*/role_section_excerpts.line.conll`.
It parses CoNLL blocks into examples, then performs tokenizer-specific subword alignment at runtime.


In [ ]:
from pathlib import Path

import torch

from training_inference.conll_data_utils import create_conll_dataset_build, summarize_conll_examples
from training_inference.data_utils import (
    PretokenizedSmellDataset,
    TokenClassificationCollator,
    create_tokenizer,
)
from training_inference.model_pipeline import SharedEncoderSmellHeads, build_optimizer
from training_inference.train_loop import make_dataloader, set_global_seed, train_model

REPO_ROOT = Path('.').resolve()
DATASET_ROOT_DIR = 'cdatasets'
CONLL_NAME = 'role_section_excerpts.line.conll'
BACKBONE_NAME = 'answerdotai/ModernBERT-base'
MAX_LENGTH = 2048
BATCH_SIZE = 2
NUM_EPOCHS = 3
SEED = 42
OUTPUT_DIR = REPO_ROOT / 'training_inference' / 'outputs_conll'

set_global_seed(SEED)


In [ ]:
dataset_build = create_conll_dataset_build(
    repo_root=REPO_ROOT,
    dataset_root_dir=DATASET_ROOT_DIR,
    conll_name=CONLL_NAME,
    seed=SEED,
)

all_examples = (
    list(dataset_build.train_examples)
    + list(dataset_build.val_examples)
    + list(dataset_build.test_examples)
)
summary = summarize_conll_examples(all_examples)
print(summary)

sample = dataset_build.train_examples[0]
print('sample id:', sample.id)
print('smell:', sample.smell_name)
print('is_detected:', sample.is_detected)
print('first 20 tokens:', sample.tokens[:20])
print('first 20 labels:', sample.labels[:20])


In [ ]:
tokenizer = create_tokenizer(BACKBONE_NAME, use_fast=True, truncation_side='right')

train_ds = PretokenizedSmellDataset(
    dataset_build.train_examples,
    tokenizer=tokenizer,
    label_maps=dataset_build.label_maps,
    smell_maps=dataset_build.smell_maps,
    max_length=MAX_LENGTH,
    label_all_subtokens=False,
    convert_b_to_i_on_subtoken=False,
)
val_ds = PretokenizedSmellDataset(
    dataset_build.val_examples,
    tokenizer=tokenizer,
    label_maps=dataset_build.label_maps,
    smell_maps=dataset_build.smell_maps,
    max_length=MAX_LENGTH,
    label_all_subtokens=False,
    convert_b_to_i_on_subtoken=False,
)

collator = TokenClassificationCollator(tokenizer=tokenizer, label_pad_id=-100)
train_loader = make_dataloader(train_ds, collator=collator, batch_size=BATCH_SIZE, shuffle=True)
val_loader = make_dataloader(val_ds, collator=collator, batch_size=BATCH_SIZE, shuffle=False)

debug_batch = next(iter(train_loader))
print('batch ids:', debug_batch['ids'][:2])
print('raw tokens[0][:20]:', debug_batch['raw_tokens'][0][:20])
print('raw labels[0][:20]:', debug_batch['raw_labels'][0][:20])
print('word_ids[0][:40]:', debug_batch['word_ids'][0][:40])
print('was_truncated:', debug_batch['was_truncated'][:2])


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SharedEncoderSmellHeads.from_hf_pretrained(
    backbone_name=BACKBONE_NAME,
    num_smells=len(dataset_build.smell_maps.smell_to_head),
    num_labels=len(dataset_build.label_maps.label_to_id),
)
optimizer = build_optimizer(model, lr=2e-5, weight_decay=0.01)

print('device:', device)
print('num_smells:', len(dataset_build.smell_maps.smell_to_head))
print('num_labels:', len(dataset_build.label_maps.label_to_id))
print('label_to_id:', dataset_build.label_maps.label_to_id)


In [ ]:
artifacts = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    id_to_label=dataset_build.label_maps.id_to_label,
    label_to_id=dataset_build.label_maps.label_to_id,
    smell_to_head=dataset_build.smell_maps.smell_to_head,
    tokenizer_name_or_path=BACKBONE_NAME,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    amp_enabled=torch.cuda.is_available(),
)

artifacts
